# Arid Ecosystem Metatranscriptomes
# QC, trimming and filtering
# September 2025

## Raw reads QC

QC of the raw reads was done using **FastQC** results were visualized using MultiQC

### Getting sample names
Getting a list of the sample names for both lane1 and lane2

In [2]:
%%bash
ls ../raw_reads_metaT/lane_1 | cut -f 1,2,3 -d '_' | sort | uniq > RNA_base_filenames_lane1.txt
ls ../raw_reads_metaT/lane_2 | cut -f 1,2,3 -d '_' | sort | uniq > RNA_base_filenames_lane2.txt
cat RNA_base_filenames_lane1.txt RNA_base_filenames_lane2.txt | cut -f1 -d '_' | sort | uniq > RNA_samplenames.txt

### FastQC script

In [3]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=fastqc
#SBATCH --nodes=1
#SBATCH --ntasks=20 
#SBATCH --time=72:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda init
source ~/.bashrc

conda activate /groups/tfaily/cayalaortiz/envs/fastqc

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"

SRR=$1

fastqc -o ${PROJECT_DIR}/fastqc/qc_lane1 -f fastq ${PROJECT_DIR}/raw_reads/${SRR}_1.fastq -t 20
    
fastqc -o ${PROJECT_DIR}/fastqc/qc_lane2 -f fastq ${PROJECT_DIR}/raw_reads/${SRR}_2.fastq -t 20' > scripts/fastqc_updated.slurm

#### Submitting script


In [4]:
%%bash

while read SRR; do
    sbatch scripts/fastqc_updated.slurm $SRR
done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15834209
Submitted batch job 15834210
Submitted batch job 15834211
Submitted batch job 15834212
Submitted batch job 15834213
Submitted batch job 15834214
Submitted batch job 15834215
Submitted batch job 15834216
Submitted batch job 15834217
Submitted batch job 15834218
Submitted batch job 15834219
Submitted batch job 15834220
Submitted batch job 15834221
Submitted batch job 15834222
Submitted batch job 15834223
Submitted batch job 15834224
Submitted batch job 15834225
Submitted batch job 15834226
Submitted batch job 15834227
Submitted batch job 15834228
Submitted batch job 15834229
Submitted batch job 15834230
Submitted batch job 15834231
Submitted batch job 15834232
Submitted batch job 15834233
Submitted batch job 15834234
Submitted batch job 15834235
Submitted batch job 15834236
Submitted batch job 15834237
Submitted batch job 15834238
Submitted batch job 15834239
Submitted batch job 15834240
Submitted batch job 15834241
Submitted batch job 15834242
Submitted batc

## Trimming using RQCfilter

RQCfilter will be used to trim the samples with **bbduk**. Additionally contaminantes from human, cat and dog will be removed.

### RQCfilter script

In [1]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=rqcfilter
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=24:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/bbtools

# Defining inputs and outputs

SRR=$1

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
OUTPUT_DIR="${PROJECT_DIR}/RNA_rqcfilter/${SRR}"

#~/bbmap/

rqcfilter2.sh in=${PROJECT_DIR}/raw_reads/fastq_reads_interleaved/fixed/${SRR}_fixed_seqkit_1.fastq \
    in2=${PROJECT_DIR}/raw_reads/fastq_reads_interleaved/fixed/${SRR}_fixed_seqkit_2.fastq \
    path=${OUTPUT_DIR} \
    rqcfilterdata=/xdisk/tfaily/vfreirezapata/metat_2025_final/databases/RQCFilterData \
    trimk=23 mink=11 qtrim=rl trimq=20 minlength=75 \
    rna=t trimfragadapter=t maxns=1 maq=10 minlen=51 mlf=0.33 phix=t removeribo=t removehuman=t removecat=t \
    removedog=t removemouse=t lambda=t detectmicrobes=t threads=94 barcodefilter=f khist=t mtst=t kapa=t clumpify=t

'> scripts/rqcfilter.slurm

In [2]:
%%bash

while read SRR 
do

    sbatch scripts/rqcfilter.slurm $SRR

done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt
    


Submitted batch job 15875606
Submitted batch job 15875607
Submitted batch job 15875608
Submitted batch job 15875609
Submitted batch job 15875610
Submitted batch job 15875611
Submitted batch job 15875612
Submitted batch job 15875613
Submitted batch job 15875614
Submitted batch job 15875615
Submitted batch job 15875616
Submitted batch job 15875617
Submitted batch job 15875618
Submitted batch job 15875619
Submitted batch job 15875620
Submitted batch job 15875621
Submitted batch job 15875622
Submitted batch job 15875623
Submitted batch job 15875624
Submitted batch job 15875625
Submitted batch job 15875626
Submitted batch job 15875627
Submitted batch job 15875628
Submitted batch job 15875629
Submitted batch job 15875630
Submitted batch job 15875631
Submitted batch job 15875632
Submitted batch job 15875633
Submitted batch job 15875634
Submitted batch job 15875635
Submitted batch job 15875636
Submitted batch job 15875637
Submitted batch job 15875638
Submitted batch job 15875639
Submitted batc

### Changing from interleaved to paired end
Reads will be changing from the interleaved format to separate paired end files using **reformat.sh**. 

In [1]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=reformat
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=10:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out

conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/bbtools


# Defining inputs and outputs
PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"

SRR=$1

INDIR="${PROJECT_DIR}/RNA_rqcfilter/${SRR}"


reformat.sh in=${INDIR}/${SRR}_fixed_seqkit_1.anqrpht.fastq.gz \
    out=${INDIR}/${SRR}_filter_1.anqrpht.fastq.gz \
    out2=${INDIR}/${SRR}_filter_2.anqrpht.fastq.gz
    
' > scripts/reformat_pe.slurm

In [2]:
%%bash
while read SRR
do
    sbatch scripts/reformat_pe.slurm $SRR

done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15878128
Submitted batch job 15878129
Submitted batch job 15878130
Submitted batch job 15878131
Submitted batch job 15878132
Submitted batch job 15878133
Submitted batch job 15878134
Submitted batch job 15878135
Submitted batch job 15878136
Submitted batch job 15878137
Submitted batch job 15878138
Submitted batch job 15878139
Submitted batch job 15878140
Submitted batch job 15878141
Submitted batch job 15878142
Submitted batch job 15878143
Submitted batch job 15878144
Submitted batch job 15878145
Submitted batch job 15878146
Submitted batch job 15878147
Submitted batch job 15878148
Submitted batch job 15878149
Submitted batch job 15878150
Submitted batch job 15878151
Submitted batch job 15878152
Submitted batch job 15878153
Submitted batch job 15878154
Submitted batch job 15878155
Submitted batch job 15878156
Submitted batch job 15878157
Submitted batch job 15878158
Submitted batch job 15878159
Submitted batch job 15878160
Submitted batch job 15878161
Submitted batc

## SortmeRna after rfqfilter

In [5]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=sortmerna
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=48:00:00 
#SBATCH --partition=standard 
#SBATCH --account=tfaily 
#SBATCH --mail-user=vfreirezapata@email.arizona.edu 
#SBATCH --mail-type=ALL 
#SBATCH -o %x-%j.out 

conda init
source ~/.bashrc
conda activate /xdisk/tfaily/vfreirezapata/env_new/sortmerna


# Defining inputs and outputs
PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"

SRR=$1

DB_DIR="${PROJECT_DIR}/dbs/rRNA_databases_v4"

rcqf_reads="${PROJECT_DIR}/RNA_rqcfilter/${SRR}"

out_dir="${PROJECT_DIR}/sortmerna_results/${SRR}"

# Make output dir
mkdir -p "$out_dir"

sortmerna -ref $DB_DIR/smr_v4.3_default_db.fasta \
    -reads ${rcqf_reads}/${SRR}_filter_1.anqrpht.fastq.gz \
    -reads ${rcqf_reads}/${SRR}_filter_2.anqrpht.fastq.gz \
    -workdir ${out_dir} \
    --threads 94 \
    -fastx -other -blast 1 -paired_in -num_alignments 1 -v
    
'> scripts/sortmerna_run.slurm

In [8]:
%%bash
while read SRR
do
    sbatch scripts/sortmerna_run.slurm $SRR
done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15880238
Submitted batch job 15880239
Submitted batch job 15880240
Submitted batch job 15880241
Submitted batch job 15880242
Submitted batch job 15880243
Submitted batch job 15880244
Submitted batch job 15880245
Submitted batch job 15880246
Submitted batch job 15880247
Submitted batch job 15880248
Submitted batch job 15880249
Submitted batch job 15880250
Submitted batch job 15880251
Submitted batch job 15880252
Submitted batch job 15880253
Submitted batch job 15880254
Submitted batch job 15880255
Submitted batch job 15880256
Submitted batch job 15880257
Submitted batch job 15880258
Submitted batch job 15880259
Submitted batch job 15880260
Submitted batch job 15880261
Submitted batch job 15880262
Submitted batch job 15880263
Submitted batch job 15880264
Submitted batch job 15880265
Submitted batch job 15880266
Submitted batch job 15880267
Submitted batch job 15880268
Submitted batch job 15880269
Submitted batch job 15880270
Submitted batch job 15880271
Submitted batc

In [1]:
%%bash

mkdir -p sortmerna_results/mrna_reads

for file in sortmerna_results/SRR*/out/other.fq.gz
do

    new_name=$(echo $file | cut -d "/" -f2)
    cp $file sortmerna_results/mrna_reads/${new_name}.sortmerna.interleaved.fq.gz

done